# SageMaker Benchmark Evaluation - Basic Usage

This notebook demonstrates the basic user-facing flow for creating and managing benchmark evaluation jobs using the BenchmarkEvaluator with Jinja2 template-based pipeline generation.

## Step 1: Discover Available Benchmarks

Discover the benchmark properties and available options:
https://docs.aws.amazon.com/sagemaker/latest/dg/nova-model-evaluation.html

In [1]:
# Configure AWS credentials and region
#! ada credentials update --provider=isengard --account=<> --role=Admin --profile=default --once
! aws configure set region us-west-2
! aws sts get-caller-identity
! aws configure get region

{
    "UserId": "AROA2TYSUERLWFIUSFEJA:aruthen-Isengard",
    "Account": "729646638167",
    "Arn": "arn:aws:sts::729646638167:assumed-role/Admin/aruthen-Isengard"
}
us-west-2


In [2]:
from sagemaker.train.evaluate import get_benchmarks, get_benchmark_properties
from rich.pretty import pprint

# Configure logging to show INFO messages
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(name)s - %(message)s'
)

# Get available benchmarks
Benchmark = get_benchmarks()
pprint(list(Benchmark))

# Print properties for a specific benchmark
pprint(get_benchmark_properties(benchmark=Benchmark.MMLU))

[
│   <_Benchmark.MMLU: 'mmlu'>,
│   <_Benchmark.MMLU_PRO: 'mmlu_pro'>,
│   <_Benchmark.BBH: 'bbh'>,
│   <_Benchmark.GPQA: 'gpqa'>,
│   <_Benchmark.MATH: 'math'>,
│   <_Benchmark.STRONG_REJECT: 'strong_reject'>,
│   <_Benchmark.IFEVAL: 'ifeval'>,
│   <_Benchmark.MMMU: 'mmmu'>,
│   <_Benchmark.LLM_JUDGE: 'llm_judge'>
]

{
│   'modality': 'Text',
│   'description': 'Multi-task Language Understanding – Tests knowledge across 57 subjects.',
│   'metrics': ['accuracy'],
│   'strategy': 'zs_cot',
│   'subtask_available': True,
│   'subtasks': [
│   │   'abstract_algebra',
│   │   'anatomy',
│   │   'astronomy',
│   │   'business_ethics',
│   │   'clinical_knowledge',
│   │   'college_biology',
│   │   'college_chemistry',
│   │   'college_computer_science',
│   │   'college_mathematics',
│   │   'college_medicine',
│   │   'college_physics',
│   │   'computer_security',
│   │   'conceptual_physics',
│   │   'econometrics',
│   │   'electrical_engineering',
│   │   'elementary_mathematics',
│   │   'formal_logic',
│   │   'global_facts',
│   │   'high_school_biology',
│   │   'high_school_chemistry',
│   │   'high_school_computer_science',
│   │   'high_school_european_history',
│   │   'high_school_geography',
│   │   'high_school_government_and_politics',
│   │   'high_school_macroeconomics',
│   │   'high_school_mathematics',
│   │   'high_school_microeconomics',
│   │   'high_school_physics',
│   │   'high_school_psychology',
│   │   'high_school_statistics',
│   │   'high_school_us_history',
│   │   'high_school_world_history',
│   │   'human_aging',
│   │   'human_sexuality',
│   │   'international_law',
│   │   'jurisprudence',
│   │   'logical_fallacies',
│   │   'machine_learning',
│   │   'management',
│   │   'marketing',
│   │   'medical_genetics',
│   │   'miscellaneous',
│   │   'moral_disputes',
│   │   'moral_scenarios',
│   │   'nutrition',
│   │   'philosophy',
│   │   'prehistory',
│   │   'professional_accounting',
│   │   'professional_law',
│   │   'professional_medicine',
│   │   'professional_psychology',
│   │   'public_relations',
│   │   'security_studies',
│   │   'sociology',
│   │   'us_foreign_policy',
│   │   'virology',
│   │   'world_religions'
│   ]
}

## Step 2: Create BenchmarkEvaluator

Create a BenchmarkEvaluator instance with the desired benchmark. The evaluator will use Jinja2 templates to render a complete pipeline definition.

**Required Parameters:**
- `benchmark`: Benchmark type from the Benchmark enum
- `base_model`: Model ARN from SageMaker hub content
- `output_s3_location`: S3 location for evaluation outputs
- `mlflow_resource_arn`: MLflow tracking server ARN for experiment tracking

**Optional Template Fields:**
These fields are used for template rendering. If not provided, defaults will be used:
- `model_package_group`: Model package group ARN
- `source_model_package`: Source model package ARN
- `dataset`: S3 URI of evaluation dataset
- `model_artifact`: ARN of model artifact for lineage tracking (auto-inferred from source_model_package)

In [3]:
from sagemaker.train.evaluate import BenchMarkEvaluator

# Create evaluator with MMLU benchmark
# These values match our successfully tested configuration
evaluator = BenchMarkEvaluator(
    benchmark=Benchmark.MMLU,
    #subtask = "abstract_algebra" # or "all"
    model="arn:aws:sagemaker:us-west-2:729646638167:model-package/sdk-test-finetuned-models/2",
    s3_output_path="s3://sagemaker-us-west-2-729646638167/model-customization/eval/",
    model_package_group="arn:aws:sagemaker:us-west-2:729646638167:model-package-group/sdk-test-finetuned-models", # Optional inferred from model if model package
    base_eval_name="mmlu-eval-demo1",
    # Note: sagemaker_session is optional and will be auto-created if not provided
    # Note: region is optional and will be auto deduced using environment variables - SAGEMAKER_REGION, AWS_REGION
)

pprint(evaluator)

[12/17/25 09:41:14] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=554116;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=659929;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/aruthen/Library/Application Support/sagemaker/config.yaml


                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=347161;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=781633;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#354\354]8;;\

                    INFO     Retrieved ModelPackage in region: us-west-2                    ]8;id=636398;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/model_resolution.py\model_resolution.py]8;;\:]8;id=78111;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/model_resolution.py#276\276]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=360101;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=180791;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[12/17/25 09:41:15] INFO     Using first available ready MLflow app:                          ]8;id=380510;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=281740;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py#136\136]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:mlflow-app/app-KPJZRTUV                      
                             NUYS                                                                                  

                    INFO     Resolved MLflow resource ARN:                                    ]8;id=849295;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=551294;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#173\173]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:mlflow-app/app-KPJZRTUV                      
                             NUYS                                                                                  

                    INFO     Model package group provided as ARN:                             ]8;id=316533;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=331551;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#205\205]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:model-package-group/sdk                      
                             -test-finetuned-models                                                                

BenchMarkEvaluator(
│   region=None,
│   sagemaker_session=<sagemaker.core.helper.session_helper.Session object at 0x13b7b8590>,
│   model='arn:aws:sagemaker:us-west-2:729646638167:model-package/sdk-test-finetuned-models/2',
│   base_eval_name='mmlu-eval-demo1',
│   s3_output_path='s3://sagemaker-us-west-2-729646638167/model-customization/eval/',
│   mlflow_resource_arn='arn:aws:sagemaker:us-west-2:729646638167:mlflow-app/app-KPJZRTUVNUYS',
│   mlflow_experiment_name=None,
│   mlflow_run_name=None,
│   networking=None,
│   kms_key_id=None,
│   model_package_group='arn:aws:sagemaker:us-west-2:729646638167:model-package-group/sdk-test-finetuned-models',
│   benchmark=<_Benchmark.MMLU: 'mmlu'>,
│   subtasks='ALL',
│   evaluate_base_model=True
)

In [11]:
# # [Optional] BASE MODEL EVAL

# from sagemaker.train.evaluate import BenchMarkEvaluator

# # Create evaluator with MMLU benchmark
# # These values match our successfully tested configuration
# evaluator = BenchMarkEvaluator(
#     benchmark=Benchmark.MMLU,
#     model="meta-textgeneration-llama-3-2-1b-instruct",
#     s3_output_path="s3://mufi-test-serverless-smtj/eval/",
#     mlflow_resource_arn="arn:aws:sagemaker:us-west-2:<>:mlflow-tracking-server/mmlu-eval-experiment",
#     # model_package_group="arn:aws:sagemaker:us-west-2:<>:model-package-group/example-name-aovqo", # Optional inferred from model if model package
#     base_eval_name="gen-qa-eval-demo",
#     # Note: sagemaker_session is optional and will be auto-created if not provided
#     # Note: region is optional and will be auto deduced using environment variables - SAGEMAKER_REGION, AWS_REGION
# )

# pprint(evaluator)

# [Optional] BASE MODEL EVAL
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks
from pprint import pprint

# Get available benchmarks
Benchmark = get_benchmarks()

# Create evaluator with configurable parameters
# This example is more general and can be easily adapted for different models and benchmarks
evaluator = BenchMarkEvaluator(
    # Choose any benchmark from: MMLU, MMLU_PRO, BBH, GPQA, MATH, STRONG_REJECT, IFEVAL, MMMU, LLM_JUDGE
    benchmark=Benchmark.MMLU,
    
    # Model can be:
    # - JumpStart model ID: "meta-textgeneration-llama-3-2-1b-instruct"
    # - ModelPackage ARN: "arn:aws:sagemaker:region:account:model-package/name/version"
    # - BaseTrainer object: trainer_instance (from training job)
    model="meta-textgeneration-llama-3-2-1b-instruct",
    
    # S3 output path for evaluation results
    s3_output_path="s3://your-bucket/eval/",
    
    # MLflow tracking (optional - will auto-resolve if not provided)
    mlflow_resource_arn="arn:aws:sagemaker:us-west-2:729646638167:mlflow-tracking-server/your-server-name",
    
    # Optional: Specify subtasks to evaluate (defaults to "ALL")
    # subtasks=["abstract_algebra", "anatomy"],  # Uncomment to run specific subtasks
    
    # Optional: Base name for the evaluation job
    base_eval_name="benchmark-eval-demo",
    
    # Optional: Whether to evaluate the base model too (defaults to True)
    evaluate_base_model=True,
    
    # Optional: Model package group (auto-inferred if using ModelPackage)
    # model_package_group="arn:aws:sagemaker:us-west-2:YOUR_ACCOUNT_ID:model-package-group/your-group",
    
    # Optional: VPC configuration for private resources
    # networking=VpcConfig(security_group_ids=["sg-xxx"], subnets=["subnet-xxx"]),
    
    # Optional: KMS key for encryption
    # kms_key_id="arn:aws:kms:us-west-2:YOUR_ACCOUNT_ID:key/key-id",
    
    # Note: sagemaker_session and region are optional and will be auto-created/deduced
)

# Print evaluator configuration
pprint(evaluator)

# Run the evaluation
execution = evaluator.evaluate()
print(f"Evaluation started: {execution}")

[12/17/25 10:15:50] INFO     Resolved MLflow resource ARN:                                    ]8;id=38460;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=741840;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#173\173]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:mlflow-tracking-server/                      
                             your-server-name                                                                      

BenchMarkEvaluator(region=None, sagemaker_session=<sagemaker.core.helper.session_helper.Session object at 0x14fede090>, model='meta-textgeneration-llama-3-2-1b-instruct', base_eval_name='benchmark-eval-demo', s3_output_path='s3://your-bucket/eval/', mlflow_resource_arn='arn:aws:sagemaker:us-west-2:729646638167:mlflow-tracking-server/your-server-name', mlflow_experiment_name=None, mlflow_run_name=None, networking=None, kms_key_id=None, model_package_group=None, benchmark=<_Benchmark.MMLU: 'mmlu'>, subtasks='ALL', evaluate_base_model=True)


                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=464786;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=199330;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[12/17/25 10:15:51] INFO     Getting or creating artifact for source:                         ]8;id=995740;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=886450;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#673\673]8;;\
                             arn:aws:sagemaker:us-west-2:aws:hub-content/SageMakerPublicHub/M                      
                             odel/meta-textgeneration-llama-3-2-1b-instruct/1.26.0                                 

                    INFO     Searching for existing artifact for base model:                  ]8;id=614278;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=627165;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#533\533]8;;\
                             arn:aws:sagemaker:us-west-2:aws:hub-content/SageMakerPublicHub/M                      
                             odel/meta-textgeneration-llama-3-2-1b-instruct/1.26.0                                 

...

                    INFO     Updating pipeline resource.                                         ]8;id=233542;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=210456;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/resources.py#30306\30306]8;;\

[12/17/25 10:15:53] INFO     Successfully updated pipeline:                                        ]8;id=181357;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=274867;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#228\228]8;;\
                             SagemakerEvaluation-BenchmarkEvaluation-499b3c7e-e456-4297-9dc0-cc573                 
                             7137c9c                                                                               

                    INFO     Starting pipeline execution: benchmark-eval-demo-1765995353           ]8;id=678871;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=351232;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#283\283]8;;\

                    INFO     Pipeline execution started:                                           ]8;id=427869;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=531509;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#294\294]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:pipeline/SagemakerEvaluation                 
                             -BenchmarkEvaluation-499b3c7e-e456-4297-9dc0-cc5737137c9c/execution/r                 
                             1ndqwb7p04x                                                                           

Evaluation started: arn='arn:aws:sagemaker:us-west-2:729646638167:pipeline/SagemakerEvaluation-BenchmarkEvaluation-499b3c7e-e456-4297-9dc0-cc5737137c9c/execution/r1ndqwb7p04x' name='benchmark-eval-demo' status=PipelineExecutionStatus(overall_status='Executing', step_details=[], failure_reason=None) last_modified_time=datetime.datetime(2025, 12, 17, 10, 15, 53, 251000, tzinfo=tzlocal()) eval_type=<EvalType.BENCHMARK: 'benchmark'> s3_output_path='s3://your-bucket/eval/' steps=[]


In [ ]:
# # [Optional] Nova testing IAD Prod

# from sagemaker.train.evaluate import BenchMarkEvaluator

# # Create evaluator with MMLU benchmark
# # These values match our successfully tested configuration
# evaluator = BenchMarkEvaluator(
#     benchmark=Benchmark.MMLU,
#     # model="arn:aws:sagemaker:us-east-1:<>:model-package/bgrv-nova-micro-sft-lora/1",
#     model="arn:aws:sagemaker:us-east-1:<>:model-package/test-nova-finetuned-models/3",
#     s3_output_path="s3://mufi-test-serverless-iad/eval/",
#     mlflow_resource_arn="arn:aws:sagemaker:us-east-1:<>:mlflow-tracking-server/mlflow-prod-server",
#     model_package_group="arn:aws:sagemaker:us-east-1:<>:model-package-group/test-nova-finetuned-models", # Optional inferred from model if model package
#     base_eval_name="gen-qa-eval-demo",
#     region="us-east-1",
#     # Note: sagemaker_session is optional and will be auto-created if not provided
#     # Note: region is optional and will be auto deduced using environment variables - SAGEMAKER_REGION, AWS_REGION
# )

# pprint(evaluator)

### Optionally update the hyperparameters

In [12]:
pprint(evaluator.hyperparameters.to_dict())

# optionally update hyperparameters
# evaluator.hyperparameters.temperature = "0.1"

# optionally get more info on types, limits, defaults.
# evaluator.hyperparameters.get_info()


[12/17/25 10:16:02] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=652240;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=66865;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

{'aggregation': '',
 'max_model_len': '12000',
 'max_new_tokens': '8192',
 'postprocessing': 'False',
 'temperature': '0',
 'top_k': '-1',
 'top_p': '1.0'}


## Step 3: Run Evaluation

Start a benchmark evaluation job. The system will:
1. Build template context with all required parameters
2. Render the pipeline definition from `DETERMINISTIC_TEMPLATE` using Jinja2
3. Create or update the pipeline with the rendered definition
4. Start the pipeline execution with empty parameters (all values pre-substituted)

**What happens during execution:**
- CreateEvaluationAction: Sets up lineage tracking
- EvaluateBaseModel & EvaluateCustomModel: Run in parallel as serverless training jobs
- AssociateLineage: Links evaluation results to lineage tracking

In [5]:
# Run evaluation with configured parameters
execution = evaluator.evaluate()
pprint(execution)

print(f"\nPipeline Execution ARN: {execution.arn}")
print(f"Initial Status: {execution.status.overall_status}")

[12/17/25 09:41:43] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=23441;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=365589;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[12/17/25 09:41:44] INFO     Getting or creating artifact for source:                         ]8;id=749390;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=765998;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#673\673]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:model-package/sdk-test-                      
                             finetuned-models/2                                                                    

                    INFO     Searching for existing artifact for model package:               ]8;id=498675;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=711145;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#533\533]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:model-package/sdk-test-                      
                             finetuned-models/2                                                                    

                    INFO     Found existing artifact:                                         ]8;id=316341;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=80898;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#542\542]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:artifact/f8b201693aa3db                      
                             f478a1eef456e4ba1b                                                                    

                    INFO     Using user-provided model_package_group ARN:                     ]8;id=743986;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=410572;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/base_evaluator.py#486\486]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:model-package-group/sdk                      
                             -test-finetuned-models                                                                

...

[12/17/25 09:41:46] INFO     Successfully updated pipeline:                                        ]8;id=472236;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=975153;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#228\228]8;;\
                             SagemakerEvaluation-BenchmarkEvaluation-499b3c7e-e456-4297-9dc0-cc573                 
                             7137c9c                                                                               

                    INFO     Starting pipeline execution: mmlu-eval-demo1-1765993306               ]8;id=980746;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=311317;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#283\283]8;;\

                    INFO     Pipeline execution started:                                           ]8;id=265623;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=214619;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#294\294]8;;\
                             arn:aws:sagemaker:us-west-2:729646638167:pipeline/SagemakerEvaluation                 
                             -BenchmarkEvaluation-499b3c7e-e456-4297-9dc0-cc5737137c9c/execution/b                 
                             2nsump3f3qj                                                                           

BenchmarkEvaluationExecution(
│   arn='arn:aws:sagemaker:us-west-2:729646638167:pipeline/SagemakerEvaluation-BenchmarkEvaluation-499b3c7e-e456-4297-9dc0-cc5737137c9c/execution/b2nsump3f3qj',
│   name='mmlu-eval-demo1',
│   status=PipelineExecutionStatus(overall_status='Executing', step_details=[], failure_reason=None),
│   last_modified_time=datetime.datetime(2025, 12, 17, 9, 41, 46, 219000, tzinfo=tzlocal()),
│   eval_type=<EvalType.BENCHMARK: 'benchmark'>,
│   s3_output_path='s3://sagemaker-us-west-2-729646638167/model-customization/eval/',
│   steps=[]
)


Pipeline Execution ARN: arn:aws:sagemaker:us-west-2:729646638167:pipeline/SagemakerEvaluation-BenchmarkEvaluation-499b3c7e-e456-4297-9dc0-cc5737137c9c/execution/b2nsump3f3qj
Initial Status: Executing


### Alternative: Override Subtasks at Runtime

For benchmarks with subtask support, you can override subtasks when calling evaluate():

In [ ]:
# Override subtasks at evaluation time
# execution = mmlu_evaluator.evaluate(subtask="abstract_algebra")  # Single subtask
# execution = mmlu_evaluator.evaluate(subtask=["abstract_algebra", "anatomy"])  # Multiple subtasks

## Step 4: Monitor Execution

Check the job status and refresh as needed:

In [6]:
# Refresh status
execution.refresh()

# Display job status with step details
pprint(execution.status)

# Display individual step statuses
if execution.status.step_details:
    print("\nStep Details:")
    for step in execution.status.step_details:
        print(f"  {step.name}: {step.status}")

PipelineExecutionStatus(
│   overall_status='Executing',
│   step_details=[
│   │   StepDetail(
│   │   │   name='EvaluateCustomModel',
│   │   │   status='Executing',
│   │   │   start_time='2025-12-17T09:41:46.876000-08:00',
│   │   │   end_time='<sagemaker.core.utils.utils.Unassigned object at 0x10a1dd9d0>',
│   │   │   display_name=None,
│   │   │   failure_reason=None
│   │   ),
│   │   StepDetail(
│   │   │   name='EvaluateBaseModel',
│   │   │   status='Executing',
│   │   │   start_time='2025-12-17T09:41:46.876000-08:00',
│   │   │   end_time='<sagemaker.core.utils.utils.Unassigned object at 0x10a1dd9d0>',
│   │   │   display_name=None,
│   │   │   failure_reason=None
│   │   ),
│   │   StepDetail(
│   │   │   name='CreateEvaluationAction',
│   │   │   status='Succeeded',
│   │   │   start_time='2025-12-17T09:41:46.876000-08:00',
│   │   │   end_time='2025-12-17T09:41:48.693000-08:00',
│   │   │   display_name=None,
│   │   │   failure_reason=None
│   │   )
│   ],
│   failure_reason=None
)


Step Details:
  EvaluateCustomModel: Executing
  EvaluateBaseModel: Executing
  CreateEvaluationAction: Succeeded


## Step 5: Wait for Completion

Wait for the pipeline to complete. This provides rich progress updates in Jupyter notebooks:

In [7]:
# Wait for job completion with progress updates
# This will show a rich progress display in Jupyter
execution.wait(target_status="Succeeded", poll=5, timeout=3600)

print(f"\nFinal Status: {execution.status.overall_status}")

╭─────────────────────────────────────────── Pipeline Execution Status ───────────────────────────────────────────╮
│  Overall Status        Succeeded                                                                                │
│  Target Status         Succeeded                                                                                │
│  Elapsed Time          1972.5s                                                                                  │
│                                                                                                                 │
│ Pipeline Steps                                                                                                  │
│  Step Name                       Status           Duration                                                      │
│  AssociateLineage                Succeeded        1.7s                                                          │
│  EvaluateCustomModel             Succeeded        1981.0s                                                       │
│  EvaluateBaseModel               Succeeded        1963.5s                                                       │
│  CreateEvaluationAction          Succeeded        1.8s                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[12/17/25 10:14:51] INFO     Final Resource Status: Succeeded                                     ]8;id=338578;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=486910;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#1002\1002]8;;\


Final Status: Succeeded


## Step 6: View Results

Display the evaluation results in a formatted table:

Output Structure:

Evaluation results are stored in S3:

```
s3://your-bucket/output/
└── job_name/
    └── output/
        └── output.tar.gz
```

Extract output.tar.gz to reveal:

```
run_name/
├── eval_results/
│   ├── results_[timestamp].json
│   ├── inference_output.jsonl
│   └── details/
│       └── model/
│           └── <execution-date-time>/
│               └── details_<task_name>_#_<datetime>.parquet
└── tensorboard_results/
    └── eval/
        └── events.out.tfevents.[timestamp]
```

In [8]:
pprint(execution.s3_output_path)
# Display results in a formatted table
execution.show_results()

's3://sagemaker-us-west-2-729646638167/model-customization/eval/'

[12/17/25 10:15:00] INFO     S3 bucket: sagemaker-us-west-2-729646638167, prefix:         ]8;id=543032;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=719272;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py#130\130]8;;\
                             model-customization/eval                                                              

                    INFO     Extracted training job name:                                  ]8;id=124836;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=892568;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py#63\63]8;;\
                             pipelines-b2nsump3f3qj-EvaluateCustomModel-nEeJSTbHVP from                            
                             step: EvaluateCustomModel                                                             

                    INFO     Extracted training job name:                                  ]8;id=584467;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=260557;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py#63\63]8;;\
                             pipelines-b2nsump3f3qj-EvaluateBaseModel-eSiBKMGsyd from                              
                             step: EvaluateBaseModel                                                               

                    INFO     Searching for results_*.json in                              ]8;id=824882;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=541269;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py#151\151]8;;\
                             s3://sagemaker-us-west-2-729646638167/model-customization/ev                          
                             al/pipelines-b2nsump3f3qj-EvaluateCustomModel-nEeJSTbHVP/out                          
                             put/output/                                                                           

...

                    INFO     Using metrics from 'all' key (standard benchmark format)      ]8;id=905637;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=620320;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py#93\93]8;;\

                    INFO     Using metrics from 'all' key (standard benchmark format)      ]8;id=315596;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=627057;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/common_utils/show_results_utils.py#93\93]8;;\

                Custom Model Results                
╭────────────────────────────────┬─────────────────╮
│ Metric                         │           Value │
├────────────────────────────────┼─────────────────┤
│ mmlu_accuracy                  │          35.22% │
│ mmlu_accuracy_stderr           │          0.0353 │
╰────────────────────────────────┴─────────────────╯

                 Base Model Results                 
╭────────────────────────────────┬─────────────────╮
│ Metric                         │           Value │
├────────────────────────────────┼─────────────────┤
│ mmlu_accuracy                  │          35.27% │
│ mmlu_accuracy_stderr           │          0.0354 │
╰────────────────────────────────┴─────────────────╯

╭─────────────────────────────────────────── Result Artifacts Location ───────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  📦 Full evaluation artifacts available at:                                                                     │
│                                                                                                                 │
│  Custom Model:                                                                                                  │
│    s3://sagemaker-us-west-2-729646638167/model-customization/eval/pipelines-b2nsump3f3qj-EvaluateCustomModel-n  │
│  EeJSTbHVP/output/output/None/eval_results/                                                                     │
│                                                                                                                 │
│  Base Model:                                                                                                    │
│    s3://sagemaker-us-west-2-729646638167/model-customization/eval/pipelines-b2nsump3f3qj-EvaluateBaseModel-eSi  │
│  BKMGsyd/output/output/None/eval_results/                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 7: Retrieve an Existing Job

You can retrieve and inspect any existing evaluation job:

In [9]:
from sagemaker.train.evaluate import EvaluationPipelineExecution
from rich.pretty import pprint


# Get an existing job by ARN
# Replace with your actual pipeline execution ARN
existing_arn = "arn:aws:sagemaker:us-west-2:<>:pipeline/SagemakerEvaluation-BenchmarkEvaluation-c344c91d-6f62-4907-85cc-7e6b29171c42/execution/inlsexrd7jes"

# base model only example
# existing_arn = "arn:aws:sagemaker:us-west-2:<>:pipeline/SagemakerEvaluation-benchmark/execution/gdp9f4dbv2vi"
existing_execution = EvaluationPipelineExecution.get(
    arn=existing_arn,
    region="us-west-2"
)

pprint(existing_execution)
print(f"\nStatus: {existing_execution.status.overall_status}")

existing_execution.show_results()

[12/17/25 10:15:05] ERROR    AWS service error when getting pipeline execution: 1 validation error ]8;id=774397;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=804681;file:///Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/train/evaluate/execution.py#767\767]8;;\
                             detected: Value                                                                       
                             'arn:aws:sagemaker:us-west-2:<>:pipeline/SagemakerEvaluation-Benchmar                 
                             kEvaluation-c344c91d-6f62-4907-85cc-7e6b29171c42/execution/inlsexrd7j                 
                             es' at 'pipelineExecutionArn' failed to satisfy constraint: Member                    
                             must satisfy regular expression pattern:                                              
                             arn:aws*:sagemaker:*:[0-9]{12}:pipeline\/.*\/execution\/.*                            

BenchmarkEvaluationExecution(
│   arn='arn:aws:sagemaker:us-west-2:<>:pipeline/SagemakerEvaluation-BenchmarkEvaluation-c344c91d-6f62-4907-85cc-7e6b29171c42/execution/inlsexrd7jes',
│   name='inlsexrd7jes',
│   status=PipelineExecutionStatus(
│   │   overall_status='Error',
│   │   step_details=[],
│   │   failure_reason="AWS service error: ValidationException:1 validation error detected: Value 'arn:aws:sagemaker:us-west-2:<>:pipeline/SagemakerEvaluation-BenchmarkEvaluation-c344c91d-6f62-4907-85cc-7e6b29171c42/execution/inlsexrd7jes' at 'pipelineExecutionArn' failed to satisfy constraint: Member must satisfy regular expression pattern: arn:aws[a-z\\-]*:sagemaker:[a-z0-9\\-]*:[0-9]{12}:pipeline\\/.*\\/execution\\/.*"
│   ),
│   last_modified_time=None,
│   eval_type=<EvalType.BENCHMARK: 'benchmark'>,
│   s3_output_path=None,
│   steps=[]
)


Status: Error


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:19                                                                                   │
│                                                                                                  │
│   16 pprint(existing_execution)                                                                  │
│   17 print(f"\nStatus: {existing_execution.status.overall_status}")                              │
│   18                                                                                             │
│ ❱ 19 existing_execution.show_results()                                                           │
│   20                                                                                             │
│                                                                                                  │
│ /Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/core │
│ /telemetry/telemetry_logging.py:175 in wrapper                                                   │
│                                                                                                  │
│   172 │   │   │   │   │   "sagemaker_session is not provided or not valid.",                     │
│   173 │   │   │   │   │   func_name,                                                             │
│   174 │   │   │   │   )                                                                          │
│ ❱ 175 │   │   │   │   return func(*args, **kwargs)                                               │
│   176 │   │                                                                                      │
│   177 │   │   return wrapper                                                                     │
│   178                                                                                            │
│                                                                                                  │
│ /Users/aruthen/amazon/sagemaker-python-sdk/venv-test/lib/python3.12/site-packages/sagemaker/trai │
│ n/evaluate/execution.py:1268 in show_results                                                     │
│                                                                                                  │
│   1265 │   │   self.refresh()                                                                    │
│   1266 │   │                                                                                     │
│   1267 │   │   if self.status.overall_status != "Succeeded":                                     │
│ ❱ 1268 │   │   │   raise ValueError(                                                             │
│   1269 │   │   │   │   f"Cannot show results. Execution status is '{self.status.overall_status}  │
│   1270 │   │   │   │   f"Results are only available after successful execution. "                │
│   1271 │   │   │   │   f"Use execution.wait() to wait for completion or check execution.status   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ValueError: Cannot show results. Execution status is 'Error'. Results are only available after successful 
execution. Use execution.wait() to wait for completion or check execution.status for details.

In [ ]:
# Run evaluation with configured parameters
execution = evaluator.evaluate()
pprint(execution)

print(f"\nPipeline Execution ARN: {execution.arn}")
print(f"Initial Status: {execution.status.overall_status}")

## Step 8: List All Benchmark Evaluations

Retrieve all benchmark evaluation executions:

In [ ]:
# Get all benchmark evaluations (returns iterator)
all_executions_iter = BenchMarkEvaluator.get_all(region="us-west-2")
all_executions = list(all_executions_iter)

print(f"Found {len(all_executions)} evaluation(s)\n")
for exec in all_executions[:5]:  # Show first 5
    print(f"  {exec.name}: {exec.status.overall_status}")

## Step 9: Stop a Running Job (Optional)

You can stop a running evaluation if needed:

In [ ]:
# Uncomment to stop the job
# existing_execution.stop()
# print(f"Execution stopped. Status: {execution.status.overall_status}")

## Understanding the Pipeline Structure

The rendered pipeline definition includes:

**4 Steps:**
1. **CreateEvaluationAction** (Lineage): Sets up tracking
2. **EvaluateBaseModel** (Training): Evaluates base model
3. **EvaluateCustomModel** (Training): Evaluates custom model
4. **AssociateLineage** (Lineage): Links results

**Key Features:**
- Template-based: Uses Jinja2 for flexible pipeline generation
- Parallel execution: Base and custom models evaluated simultaneously
- Serverless: No need to manage compute resources
- MLflow integration: Automatic experiment tracking
- Lineage tracking: Full traceability of evaluation artifacts

**Typical Execution Time:**
- Total: ~10-12 minutes
- Downloading phase: ~5-7 minutes (model)
- Training phase: ~3-5 minutes (running evaluation)
- Lineage steps: ~2-4 seconds each